# Real Estate Investment Advisor
## Phase 3: Feature Engineering

**Goal:** Build the two target variables, create new meaningful features, encode categoricals, scale numerics, and prepare the final dataset ready for modeling.

### What we'll do:
1. Create target variable 1 — `Good_Investment` (Classification)
2. Create target variable 2 — `Future_Price_5yr` (Regression)
3. Engineer new features
4. Encode categorical columns
5. Scale numeric columns
6. Save final model-ready dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

pd.set_option('display.max_columns', None)

# Load cleaned data
df = pd.read_csv("india_housing_cleaned.csv")

print("✅ Data loaded successfully")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Data loaded successfully
Shape: 250,000 rows × 23 columns


## Step 1: Create Target Variable 1 — Good_Investment (Classification)

A property is classified as a **Good Investment** if it meets the following multi-factor criteria:
- Price is at or below the **median price** of its city (affordable relative to market)
- Property is **Ready to Move** (no completion risk)
- BHK is **2 or more** (broader rental/resale appeal)
- Amenity Count is **3 or more** (lifestyle value)

In [4]:
# Further relaxed conditions to achieve better balance

# Condition 1: Price <= 75th percentile of city price
city_75pct = df.groupby('City')['Price_in_Lakhs'].transform(lambda x: x.quantile(0.75))
cond1 = df['Price_in_Lakhs'] <= city_75pct

# Condition 2: Ready to Move OR BHK >= 3 (either is acceptable)
cond2 = (df['Availability_Status'] == 'Ready_to_Move') | (df['BHK'] >= 3)

# Condition 3: Amenity Count >= 2
cond3 = df['Amenity_Count'] >= 2

# Condition 4: Nearby Schools >= 5 OR Nearby Hospitals >= 5
cond4 = (df['Nearby_Schools'] >= 5) | (df['Nearby_Hospitals'] >= 5)

# Combine all conditions
df['Good_Investment'] = (cond1 & cond2 & cond3 & cond4).astype(int)

print("✅ Good_Investment column updated")
print(f"\nClass Distribution:")
print(df['Good_Investment'].value_counts())
print(f"\nClass Balance:")
print(df['Good_Investment'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

✅ Good_Investment column updated

Class Distribution:
Good_Investment
0    149219
1    100781
Name: count, dtype: int64

Class Balance:
Good_Investment
0    59.69%
1    40.31%
Name: proportion, dtype: object


## Step 2: Create Target Variable 2 — Future_Price_5yr (Regression)

Predict the estimated property price after 5 years using a **location-based growth rate** approach:
- Metro cities (Bangalore, Mumbai, Delhi, Hyderabad, Pune, Chennai): **10% annual growth**
- Tier-2 cities (Kochi, Surat, Jaipur, Lucknow, etc.): **8% annual growth**
- All other cities: **6% annual growth**

Formula: `Future_Price = Current_Price × (1 + growth_rate) ^ 5`

In [5]:
# Step 2: Create Future_Price_5yr

# Define growth rates by city tier
metro_cities = ['Bangalore', 'Mumbai', 'Delhi', 'New Delhi', 'Hyderabad', 
                'Pune', 'Chennai', 'Kolkata', 'Gurgaon', 'Noida']

tier2_cities = ['Kochi', 'Surat', 'Jaipur', 'Lucknow', 'Ahmedabad', 
                'Bhopal', 'Indore', 'Coimbatore', 'Visakhapatnam', 
                'Vijayawada', 'Nagpur', 'Mysore']

def get_growth_rate(city):
    if city in metro_cities:
        return 0.10
    elif city in tier2_cities:
        return 0.08
    else:
        return 0.06

df['Growth_Rate'] = df['City'].apply(get_growth_rate)
df['Future_Price_5yr'] = (df['Price_in_Lakhs'] * (1 + df['Growth_Rate']) ** 5).round(2)

print("✅ Future_Price_5yr column created")
print(f"\nGrowth Rate Distribution:")
print(df['Growth_Rate'].value_counts())
print(f"\nFuture Price Stats:")
print(df['Future_Price_5yr'].describe())
print(f"\nSample comparison:")
print(df[['City', 'Price_in_Lakhs', 'Growth_Rate', 'Future_Price_5yr']].head(8))

✅ Future_Price_5yr column created

Growth Rate Distribution:
Growth_Rate
0.06    135101
0.08     64984
0.10     49915
Name: count, dtype: int64

Future Price Stats:
count    250000.000000
mean        363.284934
std         204.126752
min          13.380000
25%         187.910000
50%         360.560000
75%         535.410000
max         805.240000
Name: Future_Price_5yr, dtype: float64

Sample comparison:
         City  Price_in_Lakhs  Growth_Rate  Future_Price_5yr
0     Chennai          489.76         0.10            788.76
1        Pune          195.52         0.10            314.89
2    Ludhiana          183.79         0.06            245.95
3     Jodhpur          300.29         0.06            401.86
4      Jaipur          182.90         0.08            268.74
5    Durgapur          135.28         0.06            181.04
6  Coimbatore          318.12         0.08            467.42
7    Bilaspur          141.39         0.06            189.21


## Step 3: Engineer New Features

In [6]:
# Step 3: Engineer new features

# 1. Infrastructure Score — combines schools and hospitals
df['Infrastructure_Score'] = df['Nearby_Schools'] + df['Nearby_Hospitals']

# 2. Floor Ratio — how high up is the property relative to total floors
df['Floor_Ratio'] = (df['Floor_No'] / df['Total_Floors']).round(4)

# 3. Price Segment — bucket properties into Low / Mid / High
df['Price_Segment'] = pd.cut(df['Price_in_Lakhs'],
                              bins=[0, 150, 350, 500],
                              labels=['Low', 'Mid', 'High'])

# 4. Transport Score — encode accessibility as ordinal numeric
transport_map = {'Low': 1, 'Medium': 2, 'High': 3}
df['Transport_Score'] = df['Public_Transport_Accessibility'].map(transport_map)

# 5. Property Age Group
df['Age_Group'] = pd.cut(df['Age_of_Property'],
                          bins=[0, 5, 15, 25, 100],
                          labels=['New', 'Modern', 'Established', 'Old'])

print("✅ New features created")
print(f"\nInfrastructure_Score sample stats:")
print(df['Infrastructure_Score'].describe())
print(f"\nPrice_Segment distribution:")
print(df['Price_Segment'].value_counts())
print(f"\nAge_Group distribution:")
print(df['Age_Group'].value_counts())

✅ New features created

Infrastructure_Score sample stats:
count    250000.000000
mean         10.997876
std           4.061651
min           2.000000
25%           8.000000
50%          11.000000
75%          14.000000
max          20.000000
Name: Infrastructure_Score, dtype: float64

Price_Segment distribution:
Price_Segment
Mid     102320
High     76117
Low      71563
Name: count, dtype: int64

Age_Group distribution:
Age_Group
Established    73684
Modern         73472
Old            73278
New            29566
Name: count, dtype: int64


## Step 4: Encode Categorical Features

In [7]:
# Step 4: Encode categorical features

# Drop columns not needed for modeling
df.drop(columns=['Amenities', 'Growth_Rate', 'Year_Built'], inplace=True)
# Reason: Amenities replaced by Amenity_Count, Growth_Rate is derived,
# Year_Built is perfectly correlated with Age_of_Property

# Binary encoding (Yes/No columns)
binary_cols = ['Parking_Space', 'Security']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Ordinal encoding
df['Furnished_Status'] = df['Furnished_Status'].map({
    'Unfurnished': 0, 'Semi-furnished': 1, 'Furnished': 2
})

df['Availability_Status'] = df['Availability_Status'].map({
    'Under_Construction': 0, 'Ready_to_Move': 1
})

# Label encoding for high cardinality columns
le = LabelEncoder()
for col in ['State', 'City', 'Locality', 'Property_Type', 
            'Facing', 'Owner_Type']:
    df[col] = le.fit_transform(df[col])

# Encode engineered categorical features
df['Price_Segment'] = df['Price_Segment'].map({'Low': 0, 'Mid': 1, 'High': 2})
df['Age_Group'] = df['Age_Group'].map({'New': 0, 'Modern': 1, 'Established': 2, 'Old': 3})

print("✅ All categorical features encoded")
print(f"\nFinal dtypes:")
print(df.dtypes)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ All categorical features encoded

Final dtypes:
State                                int64
City                                 int64
Locality                             int64
Property_Type                        int64
BHK                                  int64
Size_in_SqFt                         int64
Price_in_Lakhs                     float64
Price_per_SqFt                     float64
Furnished_Status                     int64
Floor_No                             int64
Total_Floors                         int64
Age_of_Property                      int64
Nearby_Schools                       int64
Nearby_Hospitals                     int64
Public_Transport_Accessibility      object
Parking_Space                        int64
Security                             int64
Facing                               int64
Owner_Type                           int64
Availability_Status                  int64
Amenity_Count                        int64
Good_Investment                      int64
Futu

In [8]:
# Fix remaining unencoded columns

# Public_Transport_Accessibility — already have Transport_Score, drop original
df.drop(columns=['Public_Transport_Accessibility'], inplace=True)

# Age_Group — convert category to int
df['Age_Group'] = df['Age_Group'].astype(int)

# Price_Segment — convert to int (may still be float)
df['Price_Segment'] = df['Price_Segment'].astype(int)

print("✅ All columns now numeric")
print(f"\nFinal dtypes:")
print(df.dtypes)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nAny remaining object columns: {df.select_dtypes(include='object').columns.tolist()}")

✅ All columns now numeric

Final dtypes:
State                     int64
City                      int64
Locality                  int64
Property_Type             int64
BHK                       int64
Size_in_SqFt              int64
Price_in_Lakhs          float64
Price_per_SqFt          float64
Furnished_Status          int64
Floor_No                  int64
Total_Floors              int64
Age_of_Property           int64
Nearby_Schools            int64
Nearby_Hospitals          int64
Parking_Space             int64
Security                  int64
Facing                    int64
Owner_Type                int64
Availability_Status       int64
Amenity_Count             int64
Good_Investment           int64
Future_Price_5yr        float64
Infrastructure_Score      int64
Floor_Ratio             float64
Price_Segment             int64
Transport_Score           int64
Age_Group                 int64
dtype: object

Shape: 250,000 rows × 27 columns

Any remaining object columns: []


## Step 5: Scale Numeric Features

In [9]:
# Step 5: Scale numeric features using MinMaxScaler

# Define feature columns (exclude target variables)
target_cols = ['Good_Investment', 'Future_Price_5yr']

# Columns to scale (continuous numeric — not binary or already encoded categoricals)
scale_cols = ['Price_in_Lakhs', 'Price_per_SqFt', 'Size_in_SqFt', 
              'Floor_No', 'Total_Floors', 'Age_of_Property',
              'Nearby_Schools', 'Nearby_Hospitals', 'Infrastructure_Score',
              'Floor_Ratio', 'Future_Price_5yr']

scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[scale_cols] = scaler.fit_transform(df[scale_cols])

print("✅ Numeric features scaled using MinMaxScaler")
print(f"\nScaled columns sample (min should be 0, max should be 1):")
print(df_scaled[scale_cols].describe().loc[['min', 'max']].round(4))

✅ Numeric features scaled using MinMaxScaler

Scaled columns sample (min should be 0, max should be 1):
     Price_in_Lakhs  Price_per_SqFt  Size_in_SqFt  Floor_No  Total_Floors  \
min             0.0             0.0           0.0       0.0           0.0   
max             1.0             1.0           1.0       1.0           1.0   

     Age_of_Property  Nearby_Schools  Nearby_Hospitals  Infrastructure_Score  \
min              0.0             0.0               0.0                   0.0   
max              1.0             1.0               1.0                   1.0   

     Floor_Ratio  Future_Price_5yr  
min          0.0               0.0  
max          1.0               1.0  


## Step 6: Save Final Model-Ready Datasets

In [10]:
# Step 6: Save final datasets

# Save unscaled version (for reference and Streamlit app)
df.to_csv("india_housing_featured.csv", index=False)
print("✅ Unscaled featured dataset saved as: india_housing_featured.csv")

# Save scaled version (for modeling)
df_scaled.to_csv("india_housing_model_ready.csv", index=False)
print("✅ Scaled model-ready dataset saved as: india_housing_model_ready.csv")

# Final summary
print(f"\n=== Final Dataset Summary ===")
print(f"Total rows: {df_scaled.shape[0]:,}")
print(f"Total columns: {df_scaled.shape[1]}")
print(f"\nFeature columns: {df_scaled.shape[1] - 2}")
print(f"Target 1 — Good_Investment (Classification): {df_scaled['Good_Investment'].value_counts().to_dict()}")
print(f"Target 2 — Future_Price_5yr (Regression): min=₹{df['Future_Price_5yr'].min():.1f}L, max=₹{df['Future_Price_5yr'].max():.1f}L")

✅ Unscaled featured dataset saved as: india_housing_featured.csv
✅ Scaled model-ready dataset saved as: india_housing_model_ready.csv

=== Final Dataset Summary ===
Total rows: 250,000
Total columns: 27

Feature columns: 25
Target 1 — Good_Investment (Classification): {0: 149219, 1: 100781}
Target 2 — Future_Price_5yr (Regression): min=₹13.4L, max=₹805.2L


## Phase 3 Summary — Feature Engineering

### Target Variables Created

| Target | Type | Description | Distribution |
|--------|------|-------------|--------------|
| `Good_Investment` | Classification (0/1) | 1 if price ≤ 75th percentile of city + Ready to Move OR BHK≥3 + Amenities≥2 + Schools or Hospitals≥5 | 60% No / 40% Yes |
| `Future_Price_5yr` | Regression (float) | Current price compounded at city-tier growth rate for 5 years | ₹13.4L – ₹805.2L |

### New Features Engineered

| Feature | Description |
|---------|-------------|
| `Infrastructure_Score` | Nearby_Schools + Nearby_Hospitals (range: 2–20) |
| `Floor_Ratio` | Floor_No / Total_Floors — relative height of property |
| `Price_Segment` | Low / Mid / High price bucket (0/1/2) |
| `Transport_Score` | Ordinal encoding of Public Transport Accessibility (1/2/3) |
| `Age_Group` | Property age bucket — New / Modern / Established / Old (0/1/2/3) |

### Encoding & Scaling
- Binary columns (Parking, Security): Yes=1 / No=0
- Ordinal columns (Furnished, Availability): logically ordered encoding
- High cardinality columns (State, City, Locality, etc.): Label Encoded
- Dropped: `Amenities` (replaced), `Year_Built` (correlated with Age), `Growth_Rate` (derived)
- Scaled: All continuous numeric features using MinMaxScaler (0–1 range)

### Output Files
- `india_housing_featured.csv` — unscaled, for Streamlit app
- `india_housing_model_ready.csv` — scaled, for modeling

### Ready for Phase 4: Modeling